# lora-kernel — S4: the first adapters

This notebook is a shell. The implementation is
[`training/s4_train.py`](https://github.com/EvolvingAgentsLabs/lora-kernel/blob/main/training/s4_train.py)
and it is the same code the Colab CLI runs headlessly, so a number produced
here and a number produced from a terminal are the same number.

**Two questions, either of which can fail.**
1. *Does specialisation happen at all?* The adapter must beat the base on cases
   neither of them was trained on.
2. *Do experts differ by region?* The α-trained and β-trained adapters must each
   win on their own clinic — otherwise the pool is one expert wearing three
   names and there is nothing for acceptance to route between.

**The probe.** `val_delta` is the clinic where an unpublished rule **inverts**
and it is in no training split. An adapter that memorised the rule scores well
on `val` and collapses here. That gap is the false-promotion number and it is
reported beside any gain.

**Runtime.** *Runtime → Change runtime type → GPU*. `gemma-4-E4B-it` fits a free
T4; `gemma-4-26B-A4B-it` needs an A100.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install -U transformers peft trl bitsandbytes accelerate datasets

In [ ]:
!git clone -q -b plan-and-instrument https://github.com/EvolvingAgentsLabs/lora-kernel.git 2>/dev/null || (cd lora-kernel && git pull -q)
%cd lora-kernel

## Run it

Every arm is graded by `training/evaluate.py` and generated greedily. Results
are written to `s4_results.json` after **each** arm, so a session reclaimed
mid-run keeps what it already paid for.

In [ ]:
import argparse, json
from training.s4_train import run_all, verdict, main

args = argparse.Namespace(
    base="google/gemma-4-E4B-it",   # or "google/gemma-4-26B-A4B-it" on an A100
    n_val=120, n_delta=60, n_region=40,
    epochs=3, r=16, alpha=32, lr=2e-4, batch=4, accum=4,
    max_seq=1024, max_new_tokens=64, seed=0, four_bit=True)

summary = run_all(args)
print(verdict(summary))

## Take the numbers back

`s4_results.json` goes into `docs/EXPERIMENT_PLAN.md` §S4 — whichever way it
came out. A large false-promotion gap is a result, not a run to repeat until it
looks smaller.

In [ ]:
print(json.dumps({k: v for k, v in summary.items() if k not in ('regions',)}, default=str)[:1500])
!zip -qr adapters.zip adapters s4_results.json && echo 'adapters.zip ready'